In [2]:

import os, json
from PIL import Image
import numpy as np
from tqdm import tqdm

input_ann_dir = "same-object-annotations"
image_dir = "same-object-images"
output_json_path = "same-object-detection-coco.json"

coco = {
    "images": [],
    "annotations": [],
    "categories": [{"id": 1, "name": "object"}]
}

ann_id = 1
for i, filename in enumerate(tqdm(os.listdir(input_ann_dir))):
    if not filename.endswith(".json"):
        continue

    ann_path = os.path.join(input_ann_dir, filename)
    with open(ann_path) as f:
        data = json.load(f)

    img_filename = data["imagePath"]
    img_path = os.path.join(image_dir, img_filename)
    img = Image.open(img_path)
    width, height = img.size
    image_id = i + 1

    coco["images"].append({
        "file_name": img_filename,
        "id": image_id,
        "width": width,
        "height": height
    })

    for shape in data["shapes"]:
        points = np.array(shape["points"])
        xmin = float(np.min(points[:, 0]))
        ymin = float(np.min(points[:, 1]))
        xmax = float(np.max(points[:, 0]))
        ymax = float(np.max(points[:, 1]))

        coco["annotations"].append({
            "id": ann_id,
            "image_id": image_id,
            "category_id": 1,
            "bbox": [xmin, ymin, xmax - xmin, ymax - ymin],
            "area": (xmax - xmin) * (ymax - ymin),
            "iscrowd": 0
        })
        ann_id += 1

# Save
with open(output_json_path, "w") as f:
    json.dump(coco, f)

print(" COCO detection format saved to:", output_json_path)

100%|██████████| 3323/3323 [00:01<00:00, 2833.97it/s]


 COCO detection format saved to: same-object-detection-coco.json
